# Task 10: Wasserstein GAN with Gradient Penalty (WGAN-GP) for Stable Image Synthesis

**Objective:** Stabilize adversarial generative networks by implementing Wasserstein distance metrics with strict Lipschitz-1 gradient constraints.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Generator Network Architecture
class Generator(nn.Module):
    def __init__(self, z_dim=100, im_channels=1, hidden_dim=64):
        super().__init__()
        self.gen = nn.Sequential(
            nn.ConvTranspose2d(z_dim, hidden_dim * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(hidden_dim * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(hidden_dim * 4, hidden_dim * 2, 3, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_dim * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(True),
            nn.ConvTranspose2d(hidden_dim, im_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x): return self.gen(x)

# Critic Network Architecture (No sigmoid activation at the output!)
class Critic(nn.Module):
    def __init__(self, im_channels=1, hidden_dim=64):
        super().__init__()
        self.critic = nn.Sequential(
            nn.Conv2d(im_channels, hidden_dim, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_dim, hidden_dim * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_dim * 2, hidden_dim * 4, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_dim * 4, 1, 4, 1, 0)
        )
    def forward(self, x): return self.critic(x)

# Gradient Penalty calculation
def compute_gradient_penalty(critic, real_images, fake_images, device):
    batch_size = real_images.shape[0]
    epsilon = torch.rand(batch_size, 1, 1, 1, device=device).expand_as(real_images)
    
    # Interpolated sample space
    interpolated = epsilon * real_images + (1 - epsilon) * fake_images
    interpolated.requires_grad_(True)
    
    # Critic score of interpolates
    interpolated_logits = critic(interpolated)
    
    # Gradient calculation
    gradients = torch.autograd.grad(
        outputs=interpolated_logits,
        inputs=interpolated,
        grad_outputs=torch.ones_like(interpolated_logits),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]
    
    gradients = gradients.view(batch_size, -1)
    grad_norm = gradients.norm(2, dim=1)
    gp = torch.mean((grad_norm - 1) ** 2)
    return gp

In [ ]:
# Setup models and test a backward step of Critic and Generator
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gen = Generator().to(device)
critic = Critic().to(device)

opt_critic = optim.Adam(critic.parameters(), lr=1e-4, betas=(0.0, 0.9))
opt_gen = optim.Adam(gen.parameters(), lr=1e-4, betas=(0.0, 0.9))

real = torch.randn(8, 1, 28, 28, device=device) # mock inputs
noise = torch.randn(8, 100, 1, 1, device=device)

# Forward passes
fake = gen(noise)
critic_real = critic(real)
critic_fake = critic(fake.detach())

# GP Calculation
gp = compute_gradient_penalty(critic, real, fake.detach(), device)

# Critic Loss (Wasserstein Distance objective with gradient penalty)
loss_critic = critic_fake.mean() - critic_real.mean() + 10.0 * gp

opt_critic.zero_grad()
loss_critic.backward()
opt_critic.step()

# Generator Loss
critic_fake_new = critic(fake)
loss_gen = -critic_fake_new.mean()

opt_gen.zero_grad()
loss_gen.backward()
opt_gen.step()

print(f"Critic Step Loss:    {loss_critic.item():.4f}")
print(f"Generator Step Loss: {loss_gen.item():.4f}")
print("Success: Generative adversarial networks forward and backward passes executed smoothly!")